In [7]:
import copy
import os
import sys
import yaml

sys.path.append('..')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Surpress tensorflow debugging warnings

from collections import OrderedDict
from glob import glob
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, RandomSampler

import numpy as np
from scipy.ndimage import center_of_mass
import matplotlib.pyplot as plt
from PIL import Image

# Pytorch StarDist3D
from pytorch_stardist.data.utils import normalize
from pytorch_stardist.models.config import Config3D as Config3D
from pytorch_stardist.models.stardist3d import StarDist3D as StarDist3D
from utils import seed_all, prepare_conf

from stardist_tools.matching import matching_dataset
from evaluate import evaluate

import tifffile as tfl

# Need this even when not using multiprocessing
os.environ["LOCAL_RANK"] = '0'
os.environ["RANK"] = '0'

In [8]:
class BlastospimDataset(Dataset):
    def __init__(self, img_names, source_dir):
        self.img_paths = []
        #self.mask_paths = []
        for name in img_names:
            self.img_paths.append(f'{source_dir}/cropped_raw_ch_0_tp_{name}.tif')
            #self.mask_paths.append(f'{source_dir}/{name}/masks/{name}.npy')

    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        #mask_path = self.mask_paths[idx]
        
        img = tfl.imread(img_path)
        #mask = np.load(mask_path)
        
        img = img >> 4
        img = img.astype(dtype=np.uint8)

        # Make image dimensions divisible by n:
        img = img[:img.shape[0]-img.shape[0]%4, :img.shape[1]-img.shape[1]%16, :img.shape[2]-img.shape[2]%16]
        #mask = mask[:mask.shape[0]-mask.shape[0]%4, :mask.shape[1]-mask.shape[1]%16, :mask.shape[2]-mask.shape[2]%16]
        #assert img.shape == mask.shape

        # Normalize image
        n_channel = 1
        axis_norm = (0, 1, 2)  # normalize channels independently
        #img = np.expand_dims(img, 0)
        img = np.expand_dims(normalize(img, 2, 98, axis=axis_norm), 0) # Add channel for one color

        return {
            'img':img,
            #'mask':mask
        }

In [9]:
alex_testset = list(range(1,34))
#alex_testset.remove(20)
#alex_testset.remove(27)

#alex_testset = ['tp_10', 'tp_27', 'tp_41', 'tp_57', 'tp_68', 'tp_71']
#alex_testset = ['tp_20']

# tp 20 1_1_1
source_dir = '/mnt/ceph/users/ajacinto/nuclear_segmentation/Aggregate_caax/test/2025-12-18_182852/'

In [10]:
test_names = []
for testset in [alex_testset]:
    test_names += testset

test_dataset = BlastospimDataset(test_names, source_dir)
#test_dataset = BlastospimDataset(testset32, source_dir)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)

In [11]:
config_file = '/mnt/ceph/users/ajacinto/nuclear_segmentation/confs/train_convnext_unet_large.yaml'

with open(config_file) as yconf:
    opt = yaml.safe_load(yconf)

Config = Config3D
StarDist = StarDist3D

conf = Config(**opt, allow_new_params=True)

# Set random seed
seed_all(conf.random_seed)

# process the configuration variables
opt = prepare_conf(conf)

# Model instanciation
model = StarDist(opt)
model.net.load_state_dict(torch.load('/mnt/home/ajacinto/ceph/nuclear_segmentation/experiments/checkpoints/Aggregate_caax_training_0528_2026/best.pth')['model_state_dict'])
model.net.to(model.device)

model.thresholds['prob'] = 0.23
print(model.thresholds['prob'])
print(model.thresholds['nms'])
model.thresholds['nms'] = 0.28
print(sum(p.numel() for p in model.parameters()))

0.23
0.3
199250273


In [12]:
# alex_testset

# Dataset
test_dataset = BlastospimDataset(alex_testset, source_dir)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)

# Evaluate
predicted_labels = []
#gt_masks = []
counter = 0
for batch in tqdm(test_dataloader):
    #img, mask = batch['img'][0].numpy(), batch['mask'][0].numpy()
    img = batch['img'][0].numpy()
    labels, details =  model.predict_instance(img, patch_size=[128, 128, 128], context=[64, 128, 128])
    #labels, details =  model.predict_instance(img) 
    predicted_labels.append(labels)
    #gt_masks.append(mask)
    #tfl.imwrite(f'/mnt/home/ajacinto/ceph/nuclear_segmentation/7-12_and_7-15/final/mask_{alex_testset[counter]}.tif', mask)
    tfl.imwrite(f'/mnt/home/ajacinto/ceph/nuclear_segmentation/Aggregate_caax/predictions_2025-12-18_182852/{alex_testset[counter]}.tif', labels)
    counter = counter + 1
#stats = matching_dataset(gt_masks, predicted_labels, thresh=0.1, show_progress=True)
#print(stats)
#my_file = my_file.numpy()
#labels, details =  model.predict_instance(test_dataloader, patch_size=[16, 256, 256], context=[8, 128, 128])

100%|██████████| 33/33 [10:19<00:00, 18.78s/it]


In [ ]:
plt.imshow(predicted_labels[0][0])
plt.show()
#plt.imshow(gt_masks[0][0])
#plt.show()
#print(gt_masks[0].shape)